In [1]:
import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import (PatternFill, Font, Alignment, 
                              Border, Side)
from openpyxl.utils import get_column_letter

# all our model outputs hardcoded here so this notebook is self-contained
fcf_historical = {2023: 6313.0, 2024: 9498.0, 
                  2025: 12434.0, 2026: 14402.0}

fcf_projected = {2027: 15841.8, 2028: 17385.7, 2029: 18830.0,
                 2030: 20184.9, 2031: 21355.7}

revenue_historical = {2023: 31352.0, 2024: 34857.0,
                      2025: 37895.0, 2026: 41525.0}

revenue_projected = {2027: 45262.2, 2028: 48973.8, 2029: 52597.8,
                     2030: 56069.3, 2031: 59321.3}

wacc            = 0.0897
terminal_growth = 0.025
implied_price   = 344.67
current_price   = 183.26
shares          = 819.0

print("Model outputs loaded")

Model outputs loaded


In [2]:
# create workbook and set up styles
wb = Workbook()
ws = wb.active
ws.title = "DCF Model"

# define colours - navy header, light blue sections, white body
navy       = PatternFill("solid", fgColor="1F3864")
light_blue = PatternFill("solid", fgColor="D6E4F0")
green_fill = PatternFill("solid", fgColor="E8F5E9")
white      = PatternFill("solid", fgColor="FFFFFF")

# define fonts
header_font  = Font(name="Calibri", bold=True, color="FFFFFF", size=11)
section_font = Font(name="Calibri", bold=True, color="1F3864", size=10)
body_font    = Font(name="Calibri", size=10)
title_font   = Font(name="Calibri", bold=True, color="1F3864", size=14)

# thin border for cells
thin = Side(style="thin", color="BFBFBF")
border = Border(left=thin, right=thin, top=thin, bottom=thin)

# column widths
ws.column_dimensions['A'].width = 30
for col in ['B','C','D','E','F','G','H','I']:
    ws.column_dimensions[col].width = 14

print("Workbook created, styles defined")

Workbook created, styles defined


In [3]:
# title section
ws['A1'] = "Salesforce, Inc. (CRM) — DCF Valuation Model"
ws['A1'].font = title_font
ws['A2'] = "Prepared using Python | Source: yfinance, SEC 10-K filings"
ws['A2'].font = Font(name="Calibri", italic=True, 
                     color="808080", size=9)
ws.merge_cells('A1:I1')
ws.merge_cells('A2:I2')

# spacing row
ws.row_dimensions[3].height = 8

# header row - years
headers = ['', '2023', '2024', '2025', '2026', 
           '2027E', '2028E', '2029E', '2030E', '2031E']
for col, val in enumerate(headers, start=1):
    cell = ws.cell(row=4, column=col, value=val)
    cell.fill = navy
    cell.font = header_font
    cell.alignment = Alignment(horizontal='center')

# label historical vs projected
ws['B3'] = " Historical "
ws['B3'].font = Font(name="Calibri", bold=True, 
                     color="1F3864", size=9)
ws['B3'].alignment = Alignment(horizontal='center')
ws.merge_cells('B3:E3')

ws['F3'] = " Projected "
ws['F3'].font = Font(name="Calibri", bold=True,
                     color="1F3864", size=9)
ws['F3'].alignment = Alignment(horizontal='center')
ws.merge_cells('F3:J3')

print("Headers built")

Headers built


In [4]:
# revenue rows
ws['A5'] = "REVENUE"
ws['A5'].font = section_font
ws['A5'].fill = light_blue

revenue_all = {**revenue_historical, **revenue_projected}
for col, (year, val) in enumerate(revenue_all.items(), start=2):
    cell = ws.cell(row=5, column=col, value=val)
    cell.fill = light_blue
    cell.font = section_font
    cell.number_format = '#,##0.0'
    cell.alignment = Alignment(horizontal='right')

# revenue growth row
ws['A6'] = "  Revenue Growth %"
ws['A6'].font = body_font

rev_values = list(revenue_all.values())
for i, year in enumerate(revenue_all.keys()):
    if i == 0:
        ws.cell(row=6, column=i+2, value='-')
    else:
        growth = (rev_values[i] / rev_values[i-1]) - 1
        cell = ws.cell(row=6, column=i+2, value=growth)
        cell.number_format = '0.0%'
        cell.font = body_font
        cell.alignment = Alignment(horizontal='right')

# FCF rows
ws['A8'] = "FREE CASH FLOW"
ws['A8'].font = section_font
ws['A8'].fill = light_blue

fcf_all = {**fcf_historical, **fcf_projected}
for col, (year, val) in enumerate(fcf_all.items(), start=2):
    cell = ws.cell(row=8, column=col, value=val)
    cell.fill = light_blue
    cell.font = section_font
    cell.number_format = '#,##0.0'
    cell.alignment = Alignment(horizontal='right')

# FCF margin row
ws['A9'] = "  FCF Margin %"
ws['A9'].font = body_font

fcf_values = list(fcf_all.values())
rev_values_all = list(revenue_all.values())
for i in range(len(fcf_values)):
    margin = fcf_values[i] / rev_values_all[i]
    cell = ws.cell(row=9, column=i+2, value=margin)
    cell.number_format = '0.0%'
    cell.font = body_font
    cell.alignment = Alignment(horizontal='right')

print("Financial data rows added")

Financial data rows added


In [5]:
# spacing
ws.row_dimensions[10].height = 8

# valuation summary section
ws['A11'] = "VALUATION SUMMARY"
ws['A11'].font = section_font
ws['A11'].fill = light_blue
ws.merge_cells('A11:I11')

valuation_rows = [
    ("WACC",                    f"{wacc:.2%}"),
    ("Terminal Growth Rate",    f"{terminal_growth:.2%}"),
    ("Implied Share Price",     f"${implied_price:,.2f}"),
    ("Current Market Price",    f"${current_price:,.2f}"),
    ("Upside / (Downside)",     f"{(implied_price/current_price - 1):+.1%}"),
    ("Implied WACC at Mkt Price","14.14%"),
]

for i, (label, value) in enumerate(valuation_rows, start=12):
    ws.cell(row=i, column=1, value=label).font = body_font
    cell = ws.cell(row=i, column=2, value=value)
    cell.font = Font(name="Calibri", bold=True, size=10)
    cell.alignment = Alignment(horizontal='right')
    if label == "Implied Share Price":
        cell.fill = green_fill
        ws.cell(row=i, column=1).fill = green_fill

# sensitivity table
ws['A19'] = "SENSITIVITY ANALYSIS — Implied Share Price ($)"
ws['A19'].font = section_font
ws['A19'].fill = light_blue
ws.merge_cells('A19:G19')

wacc_range = [0.0797, 0.0847, 0.0897, 0.0947, 0.0997]
tgr_range  = [0.015, 0.020, 0.025, 0.030, 0.035]
prices = [
    [357.1, 381.9, 411.3, 446.6, 489.8],
    [329.9, 350.8, 375.2, 404.0, 438.7],
    [306.4, 324.2, 344.7, 368.6, 396.9],
    [285.9, 301.1, 318.6, 338.7, 362.2],
    [267.8, 281.0, 295.9, 313.1, 332.8],
]

# column headers for sensitivity
ws['A20'] = "WACC \\ TGR"
ws['A20'].font = section_font
ws['A20'].fill = navy
ws['A20'].font = Font(name="Calibri", bold=True, 
                      color="FFFFFF", size=10)

for j, tgr in enumerate(tgr_range, start=2):
    cell = ws.cell(row=20, column=j, value=f"{tgr:.1%}")
    cell.fill = navy
    cell.font = Font(name="Calibri", bold=True, 
                     color="FFFFFF", size=10)
    cell.alignment = Alignment(horizontal='center')

for i, (w, row) in enumerate(zip(wacc_range, prices), start=21):
    ws.cell(row=i, column=1, value=f"{w:.2%}").font = section_font
    for j, price in enumerate(row, start=2):
        cell = ws.cell(row=i, column=j, value=price)
        cell.number_format = '$#,##0.0'
        cell.alignment = Alignment(horizontal='center')
        cell.font = body_font
        # highlight base case
        if w == 0.0897 and tgr_range[j-2] == 0.025:
            cell.fill = green_fill
            cell.font = Font(name="Calibri", bold=True, 
                            size=10, color="1F3864")

# save the file
wb.save('salesforce_dcf_model.xlsx')
print("Excel model saved: salesforce_dcf_model.xlsx")

Excel model saved: salesforce_dcf_model.xlsx
